### Example: An overdetermined least-squares problem 

Objective: find a solution to the equation
$${\bf G} {\bf m} = {\bf d}$$
with
${\bf G}=
\begin{bmatrix} 
1  & -1 \\ 
2 & -1 \\
1 & 1
\end{bmatrix}
$
and ${\bf d}=\begin{bmatrix}-1\\0\\2.5\end{bmatrix}$

First we import a few libraries for plotting and linear algebra.

In [1]:
using PyPlot  # use plotting functions from matplotlib (matlab-like)
using LinearAlgebra  # a lot of useful matrix-vector functions

We define the matrix and the vector using (Matlab-like) codes

In [2]:
G = [1 -1;2 -1;1 1];
d = [-1; 0; 2.5];

To have a look at the equations, we rearrange every ($i$th) equation:
$$G_{i1}\cdot m_1 + G_{i2} \cdot m_2 = d_i$$
into 
$$m_2 = \frac{d_i - G_{i1}\cdot m_1}{G_{i2}}$$
and plot the function $m_2(m_1)$ in a certain range:

In [3]:
xx = [0; 2]
f=figure()
for i in 1:3
    yy = (d[i] .- G[i, 1] * xx) ./ G[i,2]
    plot(xx, yy, label="equation $i") 
end
legend()
xlabel("m1")
ylabel("m2");

### Least-squares solution
We solve the system in the least-squares sense by  
$${\bf m} = \left({\bf G}^T {\bf G}\right)^{-1} \cdot {\bf G}^T {\bf d}$$

In [4]:
m = inv(G' * G) * (G' * d)

2-element Vector{Float64}:
 0.8214285714285714
 1.7142857142857142

We also try out the backslash (\) that we already know from Matlab:

In [5]:
m = G\d  # obviously doing the right thing

2-element Vector{Float64}:
 0.8214285714285717
 1.7142857142857144

### Model and Data resolution matrices
We compute the model resolution matrix using the generalized inverse

$${\bf G}^\dagger=({\bf G}^T {\bf G})^{-1} {\bf G}^T$$

by

$${\bf R^M} = {\bf G}^\dagger{\bf G} = ({\bf G}^T {\bf G})^{-1} {\bf G}^T {\bf G}$$

and 
$${\bf R^D} = {\bf G}{\bf G}^\dagger = {\bf G}({\bf G}^T {\bf G})^{-1} {\bf G}^T$$

In [7]:
Ginv = inv(G'*G)*G'
RM = Ginv * G
display(RM)  # fully resolved (over-determined)
RD = G * Ginv
matshow(RD)
set_cmap("bwr")
clim([-1, 1])
colorbar();
# display(diag(RD))
display(RD)

2×2 Matrix{Float64}:
 1.0  0.0
 0.0  1.0

3×3 Matrix{Float64}:
  0.357143  0.428571  -0.214286
  0.428571  0.714286   0.142857
 -0.214286  0.142857   0.928571

Q: Whats the interpretation of it?
* Equation 3 (green line) is "stronger" (more independent) than the others because it is "more perpendicular"
* of 1 and 2, the second seems to be "stronger" as it better defines the least-squares solution
* Equations 1+2 are strongly correlating while equation 3 is anticorrelating with 1 and 2

### Singular value decomposition
The matrix $\bf A$ is decomposed into data and model basis vectors ($\bf U$ and $\bf V$), weighted by singular values in the vector $\bf s$.
$${\bf A}= {\bf U} \cdot \text{diag}({\bf s}) \cdot {\bf V}^T $$

In [24]:
SVD = svd(G)
display(SVD.S)
display(SVD.U)
display(SVD.V)

2-element Vector{Float64}:
 2.6457513110645907
 1.414213562373095

3×2 Matrix{Float64}:
 -0.507093   0.316228
 -0.845154  -7.62083e-17
 -0.169031  -0.948683

2×2 adjoint(::Matrix{Float64}) with eltype Float64:
 -0.894427  -0.447214
  0.447214  -0.894427

In [9]:
matshow(SVD.U)
colorbar(orientation="horizontal");

In [10]:
# a very simple matrix
B = [1 0.1;1 -0.1]
F = svd(B)
display(F.S)
display(F.V)
display(F.U)

2-element Vector{Float64}:
 1.4142135623730951
 0.1414213562373095

2×2 adjoint(::Matrix{Float64}) with eltype Float64:
 -1.0  -0.0
 -0.0  -1.0

2×2 Matrix{Float64}:
 -0.707107  -0.707107
 -0.707107   0.707107

In [16]:
x = LinRange(0, 2, 20)
A = ones(length(x), 2)
A[:, 1] = x[:]

20-element LinRange{Float64, Int64}:
 0.0, 0.105263, 0.210526, 0.315789, …, 1.68421, 1.78947, 1.89474, 2.0

In [18]:
AI = inv(A'*A)*A'

2×20 Matrix{Float64}:
 -0.135714  -0.121429  -0.107143  …   0.107143    0.121429    0.135714
  0.185714   0.171429   0.157143     -0.0571429  -0.0714286  -0.0857143

In [20]:
RD = A*AI

20×20 Matrix{Float64}:
  0.185714      0.171429     0.157143    …  -0.0714286   -0.0857143
  0.171429      0.158647     0.145865       -0.0586466   -0.0714286
  0.157143      0.145865     0.134586       -0.0458647   -0.0571429
  0.142857      0.133083     0.123308       -0.0330827   -0.0428571
  0.128571      0.120301     0.11203        -0.0203008   -0.0285714
  0.114286      0.107519     0.100752    …  -0.0075188   -0.0142857
  0.1           0.0947368    0.0894737       0.00526316   4.16334e-17
  0.0857143     0.0819549    0.0781955       0.0180451    0.0142857
  0.0714286     0.0691729    0.0669173       0.0308271    0.0285714
  0.0571429     0.056391     0.0556391       0.043609     0.0428571
  0.0428571     0.043609     0.0443609   …   0.056391     0.0571429
  0.0285714     0.0308271    0.0330827       0.0691729    0.0714286
  0.0142857     0.0180451    0.0218045       0.0819549    0.0857143
  5.55112e-17   0.00526316   0.0105263       0.0947368    0.1
 -0.0142857    -0.0075188   -

In [22]:
s = svd(A)

LinearAlgebra.SVD{Float64, Float64, Matrix{Float64}, Vector{Float64}}
U factor:
20×2 Matrix{Float64}:
 -0.0964396  -0.420016
 -0.108632   -0.383205
 -0.120824   -0.346393
 -0.133017   -0.309581
 -0.145209   -0.272769
 -0.157401   -0.235957
 -0.169594   -0.199146
 -0.181786   -0.162334
 -0.193978   -0.125522
 -0.206171   -0.0887104
 -0.218363   -0.0518987
 -0.230556   -0.0150869
 -0.242748    0.0217248
 -0.25494     0.0585366
 -0.267133    0.0953484
 -0.279325    0.13216
 -0.291517    0.168972
 -0.30371     0.205784
 -0.315902    0.242595
 -0.328094    0.279407
singular values:
2-element Vector{Float64}:
 6.6348108358678255
 1.8296738028628123
Vt factor:
2×2 Matrix{Float64}:
 -0.768493  -0.639858
  0.639858  -0.768493

In [ ]:
s.V

2×2 adjoint(::Matrix{Float64}) with eltype Float64:
 -0.768493   0.639858
 -0.639858  -0.768493